# DART OpenAPI 실습 가이드

감사대상 회사의 개략적 정보를 수집하는 파이프라인을 **한 셀씩** 실행하며 익힙니다.

**사전 준비:**
```bash
pip install requests pandas python-dotenv
```

**`.env` 파일 생성** (이 노트북과 같은 폴더에):
```
DART_API_KEY=xxxxxxxxxxxxxxxxxxxxxxxx
```

---
## 0. 환경설정

In [1]:
# 라이브러리 임포트
import requests
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
# .env 파일에서 API 키 로드
load_dotenv()
API_KEY = os.getenv('DART_API_KEY')

# 키가 제대로 로드됐는지 확인 (앞 4자리만 보여줌)
if API_KEY:
    print(f'API 키 로드 성공: {API_KEY[:4]}...')
else:
    print('❌ API 키를 찾을 수 없습니다. .env 파일을 확인하세요.')

API 키 로드 성공: a119...


In [3]:
# DART API 기본 URL
BASE_URL = 'https://opendart.fss.or.kr/api'

---
## 1. 첫 번째 API 호출 해보기

DART API는 구조가 아주 단순합니다:
- HTTP **GET** 요청
- `crtfc_key` 파라미터에 API 키를 넣고
- 나머지 파라미터로 조건을 지정

가장 간단한 **기업개황** API(`company.json`)부터 호출해봅시다.

단, 이 API는 `corp_code`(8자리 DART 고유번호)가 필요합니다.  
삼성전자의 corp_code는 `00126380`입니다. 이걸 어떻게 알아내는지는 뒤에서 다룹니다.

In [4]:
# 가장 기본적인 API 호출 — 기업개황
res = requests.get(
    f'{BASE_URL}/company.json',
    params={
        'crtfc_key': API_KEY,
        'corp_code': '00126380'   # 삼성전자
    }
)

# HTTP 상태 확인
print(f'HTTP 상태코드: {res.status_code}')

# JSON 응답 확인
data = res.json()
data

HTTP 상태코드: 200


{'status': '000',
 'message': '정상',
 'corp_code': '00126380',
 'corp_name': '삼성전자(주)',
 'corp_name_eng': 'SAMSUNG ELECTRONICS CO,.LTD',
 'stock_name': '삼성전자',
 'stock_code': '005930',
 'ceo_nm': '전영현, 노태문',
 'corp_cls': 'Y',
 'jurir_no': '1301110006246',
 'bizr_no': '1248100998',
 'adres': '경기도 수원시 영통구  삼성로 129 (매탄동)',
 'hm_url': 'www.samsung.com/sec',
 'ir_url': '',
 'phn_no': '02-2255-0114',
 'fax_no': '031-200-7538',
 'induty_code': '264',
 'est_dt': '19690113',
 'acc_mt': '12'}

응답을 보면 `status`가 `'000'`이면 정상입니다.  
회사명, 대표자, 업종코드, 결산월 등 기본정보가 들어있습니다.

이 패턴이 **모든 DART API에 동일하게 적용**됩니다:  
`requests.get(URL, params={...})` → `.json()` → `status` 확인 → 데이터 사용

---
## 2. 에러가 나면 어떻게 되는지 확인

일부러 잘못된 `corp_code`를 넣어서 에러 응답을 확인해봅시다.  
에러 핸들링을 설계하려면, 에러가 어떤 형태로 오는지 알아야 합니다.

In [5]:
# 존재하지 않는 corp_code로 호출
res_err = requests.get(
    f'{BASE_URL}/company.json',
    params={
        'crtfc_key': API_KEY,
        'corp_code': '99999999'   # 없는 코드
    }
)

err_data = res_err.json()
print(f"status: {err_data.get('status')}")
print(f"message: {err_data.get('message')}")

status: 013
message: 조회된 데이타가 없습니다.


주요 상태코드:
- `'000'` → 정상
- `'010'` → 등록되지 않은 키
- `'013'` → 조회된 데이터 없음
- `'020'` → 요청 제한 초과 (rate limit)
- `'100'` → 부적절한 파라미터 값

---
## 3. 공통 호출 함수 만들기

매번 `requests.get()`을 쓰고, `crtfc_key`를 넣고, 에러를 확인하는 건 반복입니다.  
이걸 함수로 묶어두면 이후 코드가 훨씬 깔끔해집니다.

In [6]:
import time

def call_dart(endpoint: str, params: dict) -> dict:
    """
    DART API 호출 공통 함수.
    
    - crtfc_key를 자동으로 추가
    - rate limit(020) 시 재시도
    - 에러 시 메시지 출력
    
    Args:
        endpoint: API 엔드포인트 (예: 'company.json')
        params: 쿼리 파라미터 딕셔너리
    
    Returns:
        API 응답 딕셔너리
    """
    params['crtfc_key'] = API_KEY
    
    for attempt in range(3):   # 최대 3번 재시도
        res = requests.get(f'{BASE_URL}/{endpoint}', params=params, timeout=30)
        data = res.json()
        
        status = data.get('status', '')
        
        if status == '000':    # 정상
            return data
        elif status == '020':  # rate limit
            wait = 2 ** attempt
            print(f'⏳ 요청 제한, {wait}초 대기...')
            time.sleep(wait)
            continue
        else:
            print(f"⚠️ [{endpoint}] {status}: {data.get('message', '')}")
            return data
    
    return {}

print('call_dart() 함수 정의 완료')

call_dart() 함수 정의 완료


In [7]:
# 아까와 같은 기업개황 조회를 이제 한 줄로
data = call_dart('company.json', {'corp_code': '00126380'})

print(f"회사명: {data.get('corp_name')}")
print(f"대표자: {data.get('ceo_nm')}")
print(f"업종코드: {data.get('induty_code')}")
print(f"결산월: {data.get('acc_mt')}월")

회사명: 삼성전자(주)
대표자: 전영현, 노태문
업종코드: 264
결산월: 12월


---
## 4. corp_code는 어떻게 알아내는가?

DART API의 거의 모든 엔드포인트는 **8자리 corp_code**를 요구합니다.  
주식 종목코드(6자리, 예: 005930)와는 **다른 코드**입니다.

회사명 → corp_code 변환을 위해서는  
`corpCode.xml` 엔드포인트로 **전체 기업 목록**을 다운로드해야 합니다.  
이 파일은 ZIP으로 오고, 안에 XML이 들어있습니다.

In [8]:
import zipfile
from io import BytesIO
import xml.etree.ElementTree as ET

# 전체 기업 고유번호 다운로드 (약 20MB, 좀 걸림)
print('전체 기업 목록 다운로드 중... (약 10~20초)')

res = requests.get(
    f'{BASE_URL}/corpCode.xml',
    params={'crtfc_key': API_KEY},
    timeout=60
)

print(f'다운로드 완료: {len(res.content) / 1e6:.1f} MB')

전체 기업 목록 다운로드 중... (약 10~20초)
다운로드 완료: 3.5 MB


In [9]:
# ZIP 해제 → XML 파싱 → DataFrame 변환
zf = zipfile.ZipFile(BytesIO(res.content))
xml_data = zf.read('CORPCODE.xml').decode('utf-8')
root = ET.fromstring(xml_data)

# 각 기업의 정보를 리스트로 모음
corps = []
for corp in root.findall('list'):
    corps.append({
        'corp_code': corp.findtext('corp_code', ''),
        'corp_name': corp.findtext('corp_name', ''),
        'stock_code': corp.findtext('stock_code', '').strip(),
        'modify_date': corp.findtext('modify_date', ''),
    })

corp_df = pd.DataFrame(corps)
print(f'전체 등록 기업 수: {len(corp_df):,}개')
corp_df.head()

전체 등록 기업 수: 115,230개


,corp_code,corp_name,stock_code,modify_date
0,00434003,다코,,20170630
1,00430964,굿앤엘에스,,20170630
2,00388953,크레디피아제이십오차유동화전문회사,,20170630
3,00179984,연방건설산업,,20170630
4,00420143,브룩스피알아이오토메이션잉크,,20170630


In [10]:
# stock_code가 있는 기업 = 상장사
listed = corp_df[corp_df['stock_code'] != '']
print(f'상장사 수: {len(listed):,}개')
listed.head(10)

상장사 수: 3,946개


,corp_code,corp_name,stock_code,modify_date
1944,00260985,한빛네트,036720,20170630
1956,00264529,엔플렉스,040130,20170630
1957,00358545,동서정보기술,055000,20170630
2695,00231567,애드모바일,032600,20170630
3765,00359614,리더컴,056140,20170630
3833,00153551,허메스홀딩스,012400,20170630
3866,00344746,유티엑스,045880,20170630
3943,00261188,글로포스트,037830,20170630
3944,00268020,쏠라엔텍,030390,20170630
3945,00269287,보홍,041320,20170630


In [22]:
# 회사명으로 검색하는 함수
def find_corp_code(name: str) -> pd.DataFrame:
    """회사명에 name이 포함된 기업을 검색"""
    result = corp_df[corp_df['corp_name'].str.contains(name, na=False)]
    return result

# 테스트: '삼성' 검색
find_corp_code('삼성')

,corp_code,corp_name,stock_code,modify_date
148,00126502,삼성콘크리트공업,,20170630
276,00315179,삼성코닝마이크로옵틱스,,20170630
524,00427580,삼성프론티어제십이차유동화전문유한회사,,20170630
773,00434377,삼성중소형알짜주식형뮤추얼펀드,,20170630
909,00427526,삼성신한아하론일차유동화전문유한회사,,20170630
...,...,...,...,...
113185,01922675,삼성폐차,,20250507
114272,01910759,삼성스팩10호,0044K0,20250821
114338,01194731,삼성액티브자산운용,,20250610
114458,01786514,삼성기업인수목적9호,468510,20250617


In [23]:
# 정확히 '삼성전자'로 검색
finding = find_corp_code('삼성전자')
finding

,corp_code,corp_name,stock_code,modify_date
72046,01345812,삼성전자서비스씨에스,,20230125
82533,00252074,삼성전자판매,,20240124
105052,00258999,삼성전자서비스,,20250325
110845,00366997,삼성전자로지텍,,20250827
111913,00126380,삼성전자,005930,20251201


💡 `삼성전자`를 검색하면 여러 행이 나올 수 있습니다 (삼성전자서비스, 삼성전자판매 등).  
`stock_code`가 `005930`인 행이 우리가 원하는 삼성전자이고,  
해당 행의 `corp_code`가 `00126380`입니다.

이제 이 코드를 이후 실습에서 계속 사용합니다.

In [27]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실습할 회사를 여기서 설정하세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CORP_CODE = '00126380'   # 삼성전자
BSNS_YEAR = '2023'       # 사업연도 (2024는 아직 미공시일 수 있음)
REPRT_CODE = '11011'     # 사업보고서(연간)

# reprt_code 참고:
# '11011' = 사업보고서 (연간)
# '11012' = 반기보고서
# '11013' = 1분기보고서
# '11014' = 3분기보고서

print(f'대상: corp_code={CORP_CODE}, 연도={BSNS_YEAR}, 보고서={REPRT_CODE}')

대상: corp_code=00126380, 연도=2023, 보고서=11011


---
## 5. [1단계] 기업 파악

### 5-1. 기업개황 (`company.json`)

In [28]:
company = call_dart('company.json', {'corp_code': CORP_CODE})

# 보고 싶은 필드만 뽑아서 출력
fields = {
    'corp_name': '회사명',
    'corp_name_eng': '영문명',
    'ceo_nm': '대표자',
    'corp_cls': '법인구분',       # Y=유가증권, K=코스닥, N=코넥스, E=기타
    'induty_code': '업종코드',
    'est_dt': '설립일',
    'acc_mt': '결산월',
    'adres': '주소',
    'hm_url': '홈페이지',
}

for key, label in fields.items():
    print(f'{label}: {company.get(key, "N/A")}')

회사명: 삼성전자(주)
영문명: SAMSUNG ELECTRONICS CO,.LTD
대표자: 전영현, 노태문
법인구분: Y
업종코드: 264
설립일: 19690113
결산월: 12
주소: 경기도 수원시 영통구  삼성로 129 (매탄동)
홈페이지: www.samsung.com/sec


### 5-2. 공시 검색 (`list.json`)

특정 회사의 공시 목록을 검색합니다.  
`pblntf_ty` 파라미터로 공시 유형을 필터링할 수 있습니다:
- `A` = 정기공시
- `B` = 주요사항보고
- `C` = 발행공시
- `D` = 지분공시
- `E` = 기타공시
- **`F` = 외부감사 관련** ← 감사인 입장에서 가장 중요

먼저 필터 없이 전체 공시를, 그 다음 외부감사 관련만 따로 조회해봅시다.

In [31]:
# 전체 공시 (최근 10건)
# 전체 공시 (최근 1년)
data = call_dart('list.json', {
    'corp_code': CORP_CODE,
    'bgn_de': '20240101',    # 검색 시작일 (YYYYMMDD)
    'end_de': '20241231',    # 검색 종료일
    'page_count': '10',
})

if data.get('status') == '000':
    df = pd.DataFrame(data['list'])
    print(f"전체 공시 수: {data.get('total_count', '?')}건 (최근 10건 표시)")
    print()
    # 주요 컬럼만 출력
    display_cols = ['rcept_dt', 'report_nm', 'flr_nm', 'pblntf_ty']
    print(df[[c for c in display_cols if c in df.columns]].to_string(index=False))

전체 공시 수: 284건 (최근 10건 표시)

rcept_dt                        report_nm flr_nm
20241223              임원ㆍ주요주주특정증권등소유상황보고서    김상하
20241223 [기재정정]기타경영사항(자율공시)                 삼성전자
20241220                주식등의대량보유상황보고서(일반)   삼성물산
20241219              임원ㆍ주요주주특정증권등소유상황보고서    홍준화
20241217              임원ㆍ주요주주특정증권등소유상황보고서    송기재
20241212              임원ㆍ주요주주특정증권등소유상황보고서    강종호
20241212              임원ㆍ주요주주특정증권등소유상황보고서    김진철
20241211              임원ㆍ주요주주특정증권등소유상황보고서    한기욱
20241209              임원ㆍ주요주주특정증권등소유상황보고서    박상훈
20241209              임원ㆍ주요주주특정증권등소유상황보고서    이석림


In [32]:
# 외부감사 관련 공시만 필터링
data = call_dart('list.json', {
    'corp_code': CORP_CODE,
    'bgn_de': '20240101',
    'end_de': '20241231',
    'pblntf_ty': 'F',
    'page_count': '10',
})

if data.get('status') == '000':
    df = pd.DataFrame(data['list'])
    print(f"외부감사 관련 공시: {data.get('total_count', '?')}건")
    print()
    print(df[['rcept_dt', 'report_nm', 'flr_nm']].to_string(index=False))
else:
    print('외부감사 관련 공시 없음')

⚠️ [list.json] 013: 조회된 데이타가 없습니다.
외부감사 관련 공시 없음


---
## 6. [2단계] 재무제표 분석

### 6-1. 단일회사 전체 재무제표 (`fnlttSinglAcntAll.json`)

이 API가 **감사 업무에서 가장 중요한 API**입니다.  
BS(재무상태표), IS(손익계산서), CF(현금흐름표) 등 전 계정이 한꺼번에 내려옵니다.

핵심 파라미터:
- `fs_div`: `CFS`(연결재무제표) 또는 `OFS`(별도재무제표)
- `bsns_year`: 사업연도
- `reprt_code`: 보고서 종류

In [33]:
# 연결재무제표 조회
data = call_dart('fnlttSinglAcntAll.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
    'fs_div': 'CFS',          # 연결재무제표
})

fs = pd.DataFrame(data.get('list', []))
print(f'전체 계정 수: {len(fs)}개')
print(f'컬럼: {list(fs.columns)}')
fs.head(3)

전체 계정 수: 176개
컬럼: ['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_amount', 'ord', 'currency', 'thstrm_add_amount']


,rcept_no,reprt_code,bsns_year,corp_code,sj_div,sj_nm,account_id,account_nm,account_detail,thstrm_nm,thstrm_amount,frmtrm_nm,frmtrm_amount,bfefrmtrm_nm,bfefrmtrm_amount,ord,currency,thstrm_add_amount
0,20240312000736,11011,2023,00126380,BS,재무상태표,ifrs-full_Assets,자산총계,-,제 55 기,455905980000000,제 54 기,448424507000000,제 53 기,426621158000000,7,KRW,NaN
1,20240312000736,11011,2023,00126380,BS,재무상태표,ifrs-full_CurrentAssets,유동자산,-,제 55 기,195936557000000,제 54 기,218470581000000,제 53 기,218163185000000,8,KRW,NaN
2,20240312000736,11011,2023,00126380,BS,재무상태표,dart_ShortTermOtherReceivables,미수금,-,제 55 기,6633248000000,제 54 기,6149209000000,제 53 기,4497257000000,9,KRW,NaN


응답의 주요 컬럼:
- `sj_div`: 재무제표 구분 (BS/IS/CIS/CF/SCE)
- `account_nm`: 계정과목명
- `thstrm_amount`: **당기** 금액
- `frmtrm_amount`: **전기** 금액
- `bfefrmtrm_amount`: 전전기 금액

In [34]:
# sj_div 값별로 몇 개 계정이 있는지 확인
sj_labels = {
    'BS': '재무상태표',
    'IS': '손익계산서',
    'CIS': '포괄손익계산서',
    'CF': '현금흐름표',
    'SCE': '자본변동표',
}

for code, label in sj_labels.items():
    count = len(fs[fs['sj_div'] == code])
    if count > 0:
        print(f'{label} ({code}): {count}개 계정')

재무상태표 (BS): 52개 계정
손익계산서 (IS): 18개 계정
포괄손익계산서 (CIS): 13개 계정
현금흐름표 (CF): 39개 계정
자본변동표 (SCE): 54개 계정


In [35]:
# 재무상태표(BS)만 따로 보기
bs = fs[fs['sj_div'] == 'BS'][['account_nm', 'thstrm_amount', 'frmtrm_amount']]
bs

,account_nm,thstrm_amount,frmtrm_amount
0,자산총계,455905980000000,448424507000000
1,유동자산,195936557000000,218470581000000
2,미수금,6633248000000,6149209000000
3,선급비용,3366130000000,2867823000000
4,현금및현금성자산,69080893000000,49680710000000
5,단기상각후원가금융자산,608281000000,414610000000
6,단기당기손익-공정가치금융자산,27112000000,29080000000
7,매출채권,36647393000000,35721563000000
8,재고자산,51625874000000,52187866000000
9,매각예정분류자산,217864000000,0


In [36]:
# 손익계산서(IS)만 따로 보기
income = fs[fs['sj_div'] == 'IS'][['account_nm', 'thstrm_amount', 'frmtrm_amount']]
income

,account_nm,thstrm_amount,frmtrm_amount
52,영업이익,6566976000000,43376630000000
53,기타이익,1180448000000,1962071000000
54,기타손실,1083327000000,1790176000000
55,판매비와관리비,71979938000000,68812960000000
56,매출원가,180388580000000,190041770000000
57,기본주당이익(손실),2131,8057
58,희석주당이익(손실),2131,8057
59,금융비용,12645530000000,19027689000000
60,금융수익,16100148000000,20828995000000
61,매출총이익,78546914000000,112189590000000


In [37]:
# 현금흐름표(CF)만 따로 보기
cf = fs[fs['sj_div'] == 'CF'][['account_nm', 'thstrm_amount', 'frmtrm_amount']]
cf

,account_nm,thstrm_amount,frmtrm_amount
83,기초현금및현금성자산,49680710000000,39031415000000
84,기말현금및현금성자산,69080893000000,49680710000000
85,매각예정분류,-14153000000,0
86,재무활동현금흐름,-8593059000000,-19390049000000
87,장기차입금의 차입 (주27),354712000000,271997000000
88,사채 및 장기차입금의 상환 (주27),1219579000000,1508465000000
89,비지배지분의 증감,-9118000000,-6000000
90,단기차입금의 순증가(감소) (주27),2145400000000,-8339149000000
91,배당금의 지급,9864474000000,9814426000000
92,투자활동현금흐름,-16922817000000,-31602804000000


### 6-2. 핵심 계정 전기 대비 변동 분석

금액이 문자열(쉼표 포함)로 오기 때문에, 숫자로 변환해야 계산이 가능합니다.

In [38]:
def to_num(val) -> int:
    """DART 금액 문자열을 정수로 변환. 빈 값이면 0 반환."""
    if pd.isna(val) or val == '':
        return 0
    return int(str(val).replace(',', ''))

# 보고 싶은 핵심 계정 목록
key_accounts = [
    '자산총계', '부채총계', '자본총계',
    '매출액', '영업이익', '당기순이익',
    '수익(매출액)',     # 회사에 따라 이 이름일 수 있음
]

print(f"{'계정명':<20} {'당기(억)':>12} {'전기(억)':>12} {'변동률':>10}")
print('-' * 58)

for acct_name in key_accounts:
    row = fs[fs['account_nm'] == acct_name]
    if row.empty:
        continue
    
    row = row.iloc[0]
    curr = to_num(row.get('thstrm_amount', ''))
    prev = to_num(row.get('frmtrm_amount', ''))
    
    # 변동률 계산
    if prev != 0:
        change = f"{((curr - prev) / abs(prev)) * 100:+.1f}%"
    else:
        change = 'N/A'
    
    print(f"{acct_name:<20} {curr/1e8:>12,.0f} {prev/1e8:>12,.0f} {change:>10}")

계정명                         당기(억)        전기(억)        변동률
----------------------------------------------------------
자산총계                    4,559,060    4,484,245      +1.7%
부채총계                      922,281      936,749      -1.5%
자본총계                    3,636,779    3,547,496      +2.5%
영업이익                       65,670      433,766     -84.9%


### 6-3. (참고) 다중회사 비교 (`fnlttMultiAcnt.json`)

여러 회사의 주요 계정을 한번에 조회할 수 있습니다.  
`corp_code`를 쉼표로 구분하여 **최대 약 20개**까지 동시 조회 가능합니다.

In [39]:
# 삼성전자 vs SK하이닉스 비교 예시
# (SK하이닉스 corp_code는 find_corp_code('SK하이닉스')로 확인)

data = call_dart('fnlttMultiAcnt.json', {
    'corp_code': '00126380,00164779',   # 삼성전자, SK하이닉스
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

if data.get('status') == '000':
    multi = pd.DataFrame(data['list'])
    # 회사명 + 계정명 + 당기금액만 보기
    cols = ['corp_name', 'account_nm', 'thstrm_amount']
    available = [c for c in cols if c in multi.columns]
    print(multi[available].to_string(index=False))

account_nm       thstrm_amount
      유동자산 195,936,557,000,000
     비유동자산 259,969,423,000,000
      자산총계 455,905,980,000,000
      유동부채  75,719,452,000,000
     비유동부채  16,508,663,000,000
      부채총계  92,228,115,000,000
       자본금     897,514,000,000
     이익잉여금 346,652,238,000,000
      자본총계 363,677,865,000,000
       매출액 258,935,494,000,000
      영업이익   6,566,976,000,000
법인세차감전 순이익  11,006,265,000,000
 당기순이익(손실)  15,487,100,000,000
     총포괄손익  18,837,411,000,000
      유동자산  68,548,442,000,000
     비유동자산 228,308,847,000,000
      자산총계 296,857,289,000,000
      유동부채  41,775,101,000,000
     비유동부채  30,294,414,000,000
      부채총계  72,069,515,000,000
       자본금     897,514,000,000
     이익잉여금 219,963,351,000,000
      자본총계 224,787,774,000,000
       매출액 170,374,090,000,000
      영업이익 -11,526,297,000,000
법인세차감전 순이익  17,531,500,000,000
 당기순이익(손실)  25,397,099,000,000
     총포괄손익  25,181,020,000,000
      유동자산  30,468,100,000,000
     비유동자산  69,862,065,000,000
      자산총계 100,330,165,000,000
      유동

---
## 7. [3단계] 감사인 정보

### 7-1. 감사의견 (`accnutAdtorNmNdAdtOpinion.json`)

In [40]:
data = call_dart('accnutAdtorNmNdAdtOpinion.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

if data.get('status') == '000':
    for item in data.get('list', []):
        print(f"감사인: {item.get('adtor', 'N/A')}")
        print(f"감사의견: {item.get('adt_opinion', 'N/A')}")
        print(f"강조사항: {item.get('emphs_mtr', '없음')}")
        print(f"핵심감사사항: {item.get('core_adt_mtr', '없음')}")

감사인: 삼정회계법인
감사의견: 적정
강조사항: 없음
핵심감사사항: 없음
감사인: 안진회계법인
감사의견: 적정
강조사항: 없음
핵심감사사항: 없음
감사인: 안진회계법인
감사의견: 적정
강조사항: 없음
핵심감사사항: 없음


### 7-2. 감사용역 계약 (`adtServCnclsSttus.json`)

In [43]:
data = call_dart('adtServcCnclsSttus.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

audit_contract = pd.DataFrame(data.get('list', []))

if not audit_contract.empty:
    for _, row in audit_contract.iterrows():
        print(f"사업연도: {row.get('bsns_year_at', '')}")
        print(f"감사인: {row.get('adtor', '')}")
        print(f"내용: {row.get('cn', '')}")
        print(f"보수(천원): {row.get('mendng_amt', '')}")
        print(f"시간(시간): {row.get('mendng_tm', '')}")
        print()

사업연도: 
감사인: 삼정회계법인
내용: 분ㆍ반기 재무제표 검토
별도 및 연결 재무제표에 대한 감사
별도 및 연결 내부회계관리제도 감사
보수(천원): 
시간(시간): 

사업연도: 
감사인: 안진회계법인
내용: 분ㆍ반기 재무제표 검토
별도 및 연결 재무제표에 대한 감사
내부회계관리제도 감사
보수(천원): 
시간(시간): 

사업연도: 
감사인: 안진회계법인
내용: 분ㆍ반기 재무제표 검토
별도 및 연결 재무제표에 대한 감사
내부회계관리제도 감사
보수(천원): 
시간(시간): 



### 7-3. 비감사용역 계약 (`accnutAdtorNonAdtServCnclsSttus.json`)

감사인이 감사 외에 자문·세무·컨설팅 등 비감사용역을 수행한 경우,  
**감사인 독립성**에 영향을 줄 수 있어 감사계획 수립 시 반드시 확인해야 합니다.

In [44]:
data = call_dart('accnutAdtorNonAdtServcCnclsSttus.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

non_audit = pd.DataFrame(data.get('list', []))

if not non_audit.empty:
    for _, row in non_audit.iterrows():
        print(f"용역내용: {row.get('srvs_cn', '')}")
        print(f"용역기간: {row.get('srvs_pd', '')}")
        print(f"보수(천원): {row.get('mendng_amt', '')}")
        print()
else:
    print('비감사용역 계약 없음')

용역내용: 
용역기간: 
보수(천원): 

용역내용: 
용역기간: 
보수(천원): 

용역내용: 
용역기간: 
보수(천원): 



---
## 8. [4단계] 지배구조

### 8-1. 최대주주 현황 (`hyslrSttus.json`)

In [45]:
data = call_dart('hyslrSttus.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

shareholders = pd.DataFrame(data.get('list', []))

if not shareholders.empty:
    # 어떤 컬럼이 있는지 먼저 확인
    print(f'컬럼: {list(shareholders.columns)}')
    print()
    shareholders.head(10)

컬럼: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'stock_knd', 'nm', 'relate', 'bsis_posesn_stock_co', 'bsis_posesn_stock_qota_rt', 'trmend_posesn_stock_co', 'trmend_posesn_stock_qota_rt', 'rm', 'stlm_dt']



### 8-2. 임원 현황 (`exctvSttus.json`)

In [46]:
data = call_dart('exctvSttus.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

executives = pd.DataFrame(data.get('list', []))

if not executives.empty:
    print(f'총 임원 수: {len(executives)}명')
    print(f'컬럼: {list(executives.columns)}')
    print()
    executives.head(10)

총 임원 수: 11명
컬럼: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'nm', 'sexdstn', 'birth_ym', 'ofcps', 'rgist_exctv_at', 'fte_at', 'chrg_job', 'main_career', 'mxmm_shrholdr_relate', 'hffc_pd', 'tenure_end_on', 'stlm_dt']



In [47]:
# 등기/미등기 구분
if not executives.empty and 'rgist_exctv_at' in executives.columns:
    print(executives['rgist_exctv_at'].value_counts())
    print()

# 사외이사 비율 (지배구조 핵심 지표)
if not executives.empty and 'ofcps' in executives.columns:
    outside = len(executives[executives['ofcps'].str.contains('사외이사', na=False)])
    total_dir = len(executives[executives['ofcps'].str.contains('이사', na=False)])
    if total_dir > 0:
        print(f'사외이사 비율: {outside}/{total_dir} = {outside/total_dir*100:.1f}%')

rgist_exctv_at
사외이사    6
사내이사    5
Name: count, dtype: int64

사외이사 비율: 0/6 = 0.0%


### 8-3. 고액보수자 현황 (`indvdlByPay.json`)

보수 5억 이상인 임직원의 개인별 보수가 공시됩니다.

In [48]:
data = call_dart('indvdlByPay.json', {
    'corp_code': CORP_CODE,
    'bsns_year': BSNS_YEAR,
    'reprt_code': REPRT_CODE,
})

high_pay = pd.DataFrame(data.get('list', []))

if not high_pay.empty:
    print(f'컬럼: {list(high_pay.columns)}')
    print()
    high_pay
else:
    print('5억 이상 보수자 없음 또는 데이터 미공시')

컬럼: ['rcept_no', 'corp_cls', 'corp_code', 'corp_name', 'nm', 'ofcps', 'mendng_totamt', 'mendng_totamt_ct_incls_mendng', 'stlm_dt']



---
## 9. 정리

### 이 노트북에서 호출한 API 목록

| 단계 | API | 엔드포인트 | 용도 |
|:---:|---|---|---|
| 0 | 고유번호 | `corpCode.xml` | 회사명 → corp_code 변환 |
| 1 | 기업개황 | `company.json` | 대표자, 업종, 결산월 등 기본정보 |
| 1 | 공시검색 | `list.json` | 공시 목록 (외부감사 관련 필터링) |
| 2 | 전체 재무제표 | `fnlttSinglAcntAll.json` | BS/IS/CF 전 계정 |
| 2 | 다중회사 비교 | `fnlttMultiAcnt.json` | 동종업체 비교 |
| 3 | 감사의견 | `accnutAdtorNmNdAdtOpinion.json` | 감사인, 적정/한정 의견 |
| 3 | 감사용역 | `adtServCnclsSttus.json` | 감사보수, 감사시간 |
| 3 | 비감사용역 | `accnutAdtorNonAdtServCnclsSttus.json` | 독립성 평가 |
| 4 | 최대주주 | `hyslrSttus.json` | 지배구조 |
| 4 | 임원현황 | `exctvSttus.json` | 사외이사 비율 등 |
| 4 | 고액보수자 | `indvdlByPay.json` | 5억 이상 보수 |

### 다음 단계

이 API들을 LangGraph의 **tool**로 감싸면,  
Agent가 "삼성전자의 감사의견 알려줘"라는 질문에  
스스로 `accnutAdtorNmNdAdtOpinion.json`을 호출하고 결과를 요약할 수 있습니다.